# Projeto de Análise de Dados - Titanic

Neste projeto, continuei a análise da base do Titanic com foco em preparar melhor os dados e testar alguns modelos de classificação para prever a sobrevivência dos passageiros.

A proposta aqui foi fazer um tratamento mais completo da base, criar algumas variáveis novas e comparar o desempenho de modelos diferentes para entender qual se ajusta melhor ao problema.


## 1. Importando as bibliotecas e carregando as bases

Para começar, importei o pandas e carreguei os arquivos de treino e teste.


In [ ]:
import pandas as pd

treino = pd.read_csv('train.csv')
teste = pd.read_csv('test.csv')

treino.head(3)


In [ ]:
teste.head(3)


## 2. Fazendo o tratamento inicial dos dados

Nesta etapa, removi colunas com alta cardinalidade e tratei os valores ausentes das colunas mais importantes para o modelo.


In [ ]:
# Removendo colunas com muitas informações únicas
treino = treino.drop(['Name', 'Ticket', 'Cabin'], axis=1)
teste = teste.drop(['Name', 'Ticket', 'Cabin'], axis=1)

# Preenchendo valores ausentes de Age com a média
treino.loc[treino['Age'].isnull(), 'Age'] = treino['Age'].mean()
teste.loc[teste['Age'].isnull(), 'Age'] = teste['Age'].mean()

# Preenchendo valores ausentes de Embarked com a moda
treino.loc[treino['Embarked'].isnull(), 'Embarked'] = treino['Embarked'].mode()[0]

# Preenchendo valores ausentes de Fare com a média
teste.loc[teste['Fare'].isnull(), 'Fare'] = teste['Fare'].mean()


## 3. Criando novas variáveis

Depois do tratamento inicial, criei algumas colunas para enriquecer a base e ajudar os modelos a captarem melhor certos padrões.


In [ ]:
# Transformando a coluna Sex em uma variável numérica
treino['MaleCheck'] = treino['Sex'].apply(lambda x: 1 if x == 'male' else 0)
teste['MaleCheck'] = teste['Sex'].apply(lambda x: 1 if x == 'male' else 0)


In [ ]:
# Fazendo a padronização robusta das colunas Age e Fare
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
scaler.fit(treino[['Age', 'Fare']])

treino[['Age', 'Fare']] = scaler.transform(treino[['Age', 'Fare']])
teste[['Age', 'Fare']] = scaler.transform(teste[['Age', 'Fare']])


In [ ]:
# Criando uma variável para identificar quem estava sozinho
def sozinho(sibsp, parch):
    if sibsp == 0 and parch == 0:
        return 1
    return 0

treino['Sozinho'] = treino.apply(lambda x: sozinho(x['SibSp'], x['Parch']), axis=1)
teste['Sozinho'] = teste.apply(lambda x: sozinho(x['SibSp'], x['Parch']), axis=1)


In [ ]:
# Criando uma variável com a quantidade de familiares
treino['Familiares'] = treino['SibSp'] + treino['Parch']
teste['Familiares'] = teste['SibSp'] + teste['Parch']


In [ ]:
# Transformando Embarked em variável numérica
from sklearn.preprocessing import OrdinalEncoder

categorias = ['S', 'C', 'Q']
encoder = OrdinalEncoder(categories=[categorias], dtype='int32')

encoder.fit(treino[['Embarked']])

treino['Embarked'] = encoder.transform(treino[['Embarked']])
teste['Embarked'] = encoder.transform(teste[['Embarked']])


In [ ]:
# Removendo a coluna original de texto
treino = treino.drop('Sex', axis=1)
teste = teste.drop('Sex', axis=1)

treino.head(3)


## 4. Separando treino e validação

Com a base pronta, separei os dados em variáveis de entrada e saída e depois fiz a divisão entre treino e validação.


In [ ]:
from sklearn.model_selection import train_test_split

X = treino.drop(['PassengerId', 'Survived'], axis=1)
y = treino['Survived']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.33, random_state=42
)


## 5. Testando modelos de classificação

Nesta etapa, comparei três modelos:
- Regressão Logística
- Random Forest
- MLPClassifier


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

modelo_rl = LogisticRegression(random_state=42, max_iter=1000)
modelo_rl.fit(X_train, y_train)
y_pred_rl = modelo_rl.predict(X_val)

modelo_rf = RandomForestClassifier(random_state=42)
modelo_rf.fit(X_train, y_train)
y_pred_rf = modelo_rf.predict(X_val)

modelo_mlp = MLPClassifier(random_state=42, max_iter=5000)
modelo_mlp.fit(X_train, y_train)
y_pred_mlp = modelo_mlp.predict(X_val)


## 6. Avaliando os resultados

Para comparar os modelos, usei a acurácia e a matriz de confusão.


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

print('Acurácia - Regressão Logística:', accuracy_score(y_val, y_pred_rl))
print('Acurácia - Random Forest:', accuracy_score(y_val, y_pred_rf))
print('Acurácia - MLPClassifier:', accuracy_score(y_val, y_pred_mlp))


In [ ]:
print('Matriz de confusão - Regressão Logística')
print(confusion_matrix(y_val, y_pred_rl))

print('\nMatriz de confusão - Random Forest')
print(confusion_matrix(y_val, y_pred_rf))

print('\nMatriz de confusão - MLPClassifier')
print(confusion_matrix(y_val, y_pred_mlp))


## 7. Fazendo a previsão na base de teste

Depois da comparação, usei o modelo com melhor desempenho para gerar a previsão final da base de teste.


In [ ]:
X_teste = teste.drop('PassengerId', axis=1)

# Ajuste aqui caso outro modelo tenha performado melhor na sua validação
y_pred = modelo_mlp.predict(X_teste)

teste['Survived'] = y_pred
base_envio = teste[['PassengerId', 'Survived']]

base_envio.to_csv('resultados8.csv', index=False)

base_envio.head()


## 8. Conclusão

Com esse projeto, consegui avançar no tratamento da base do Titanic, criar novas variáveis e comparar diferentes algoritmos de classificação.

Além de praticar a parte de modelagem, esse processo também ajudou a reforçar etapas importantes como limpeza de dados, transformação de variáveis e avaliação de desempenho dos modelos.
